# TASK OVERVIEW

This project evaluates LLMs on preference-based matching problems using 4 tasks:

1. Task 1: Generate a stable matching  
2. Task 2: Detect instability (is matching stable or not)  
3. Task 3: Resolve instability (fix an unstable matching)  
4. Task 4: Preference reasoning (queries over rankings)

Models used:
- Basic model (baseline)
- Reasoning model (expected to perform better)

Dataset:
- Small instances (n = 5) using IC preferences

=========================
COMMON SETUP AND HELPERS
=========================
Goal: Define shared functions and utilities used across all tasks.

Includes:

- dataset loading
- preference parsing
- instance preparation
- answer template generation
- shared utility functions

In [18]:
!pip install -q langchain-groq

In [19]:
import os
import re
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import sys
from getpass import getpass
from langchain_groq import ChatGroq
from IPython.display import display


In [20]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [23]:
from src.config import setup_api_key, DATA_DIR, RESULTS_DIR
from src.config import setup_api_key, DATA_DIR, RESULTS_DIR
from src.task1_stable_matching import basic_model, reasoning_model
from src.task2_detect_instability import task2_basic_model, task2_reasoning_model
from src.task3_resolve_instability import task3_basic_model, task3_reasoning_model
from src.task4_preference_reasoning import task4_basic_model, task4_reasoning_model
from src.io_utils import save_summary_json, load_summary_json, save_dataframe_csv
from src.reporting import build_final_summary_table, build_chart_dataframe, plot_grouped_bar_chart
from src.config import setup_api_key, DATA_DIR, RESULTS_DIR


In [22]:
import sys
from pathlib import Path

# Current working directory is likely .../project/notebooks
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# Add the actual project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [25]:
SUMMARIES_DIR = RESULTS_DIR / "summaries"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"

In [11]:
GROQ_API_KEY = getpass("Enter GROQ API key: ")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CSV_FILE = DATA_DIR / "5_ic_processed.csv"

# =========================
# TASK 1: GENERATE STABLE MATCHING
# =========================


Goal:
Generate a stable matching from given preference lists.

Input:
- Preference lists for men and women

Output:
- One-to-one matching in JSON format

Evaluation:
- Parsed correctly
- Valid matching (one-to-one)
- Stable (no blocking pairs)
- Exact match with ground truth
- Blocking pairs count

In [ ]:
basic_detailed, basic_results, basic_summary = basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

basic_detailed_20, basic_results_20, basic_summary_20 = basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=20,
    num_examples_to_show=2
)
save_summary_json(basic_summary, SUMMARIES_DIR / "task1_basic_10.json")
save_summary_json(basic_summary_20, SUMMARIES_DIR / "task1_basic_20.json")

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 4
blocking pairs: [('M1', 'W3'), ('M2', 'W3'), ('M3', 'W3'), ('M4', 'W3')]

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 1
blocking pairs: [('M1', 'W3')]

{'total_instances': 10, 'parsed_count': 10, 'valid_count': 10, 'stable_count': 3, 'exact_match_count': 1, 'avg_blocking_pairs': 2.0}


In [ ]:
reasoning_detailed, reasoning_results, reasoning_summary = reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)


reasoning_detailed_20, reasoning_results_20, reasoning_summary_20 = reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=20,
    num_examples_to_show=2
)


save_summary_json(reasoning_summary, SUMMARIES_DIR / "task1_reasoning_10.json")
save_summary_json(reasoning_summary_20, SUMMARIES_DIR / "task1_reasoning_20.json")

parsed: True
valid: True
stable: True
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 0

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 2
blocking pairs: [('M3', 'W5'), ('M3', 'W1')]

{'total_instances': 25, 'parsed_count': 25, 'valid_count': 24, 'stable_count': 10, 'exact_match_count': 3, 'avg_blocking_pairs': 1.36}


### Task 1 Insights: Generating Stable Matchings

Both models were able to consistently produce valid one-to-one matchings, indicating that they understood the basic structure of the problem.

However, generating *stable* matchings proved challenging. The basic model produced only a small number of stable outcomes, while the reasoning model performed better but still failed to consistently eliminate blocking pairs.

A key observation is that most outputs were close to correct, often containing only a small number of blocking pairs. This suggests that while the models partially capture the structure of the problem, they struggle to enforce global stability constraints.

Overall, the reasoning model outperformed the basic model, but neither model reliably generated stable matchings even for small instances (n = 5).

# =========================
# TASK 2: DETECT INSTABILITY
# =========================

Goal:
Given a matching, determine whether it is stable or unstable.

Output:
- YES (stable) or NO (unstable)

In [ ]:
# =========================================
# 2.7 run task 2 experiments
# =========================================

task2_basic_detailed, task2_basic_summary = task2_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 2 basic summary - 10 instance ")
print(task2_basic_summary)

print("\n" + "="*50 + "\n")



task2_basic_detailed_20, task2_basic_summary_20 = task2_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=20,
    num_examples_to_show=2
)

print()
print("task 2 basic summary - 20 instance")
print(task2_basic_summary_20)

print("\n" + "="*50 + "\n")



save_summary_json(task2_basic_summary, SUMMARIES_DIR / "task2_basic_10.json")
save_summary_json(task2_basic_summary_20, SUMMARIES_DIR / "task1_basic_20.json")


parsed: True
predicted: NO
true label: YES
correct: False
parsed: True
predicted: NO
true label: NO
correct: True

task 2 basic summary
{'total_instances': 10, 'parsed_count': 10, 'correct_count': 5, 'accuracy': 0.5}


parsed: True
predicted: NO
true label: YES
correct: False
parsed: True
predicted: NO
true label: NO
correct: True

task 2 reasoning summary
{'total_instances': 10, 'parsed_count': 10, 'correct_count': 6, 'accuracy': 0.6}


In [ ]:
task2_reasoning_detailed, task2_reasoning_summary = task2_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 2 reasoning summary - 10 instances")
print(task2_reasoning_summary)


task2_reasoning_detailed_20, task2_reasoning_summary_20 = task2_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=20,
    num_examples_to_show=2
)

print()
print("task 2 reasoning summary - 20 instances")
print(task2_reasoning_summary_20)





save_summary_json(task2_reasoning_summary, SUMMARIES_DIR / "task2_reasoning_10.json")
save_summary_json(task2_reasoning_summary_20, SUMMARIES_DIR / "task1_reasoning_20.json")

### Task 2 Insights: Detecting Instability

In this task, both models were evaluated on their ability to classify matchings as stable or unstable.

The basic model achieved an accuracy of 50%, indicating near-random performance. In contrast, the reasoning model achieved 70% accuracy, demonstrating a clear improvement in its ability to analyze stability conditions.

A notable pattern is that both models frequently predicted matchings as unstable, even when they were stable. This suggests a bias toward detecting instability, possibly due to difficulty verifying the absence of blocking pairs.

Overall, instability detection is easier than generating stable matchings, but still non-trivial. The reasoning model shows improved performance, yet still makes systematic errors.

=========================
TASK 3: RESOLVE INSTABILITY
=========================
Goal: Given an unstable matching, modify it to produce a stable matching.

Output:

A valid one-to-one stable matching in JSON format

In [ ]:
# =========================================
# 3.5 run task 3 experiments
# =========================================

task3_basic_detailed, task3_basic_summary = task3_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 3 basic summary - 10 instance")
print(task3_basic_summary)

print("\n" + "=" * 50 + "\n")



task3_basic_detailed_20, task3_basic_summary_20 = task3_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=20,
    num_examples_to_show=2
)

print()
print("task 3 basic summary - 20 instance")
print(task3_basic_summary_20)

print("\n" + "=" * 50 + "\n")



save_summary_json(task3_basic_summary, SUMMARIES_DIR / "task3_basic_10.json")
save_summary_json(task3_basic_summary_20, SUMMARIES_DIR / "task3_basic_20.json")

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 3
blocking pairs: [('M2', 'W3'), ('M3', 'W3'), ('M3', 'W1')]

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 1
blocking pairs: [('M1', 'W3')]


task 3 basic summary
{'total_instances': 10, 'parsed_count': 10, 'valid_count': 9, 'stable_count': 1, 'exact_match_count': 0, 'avg_blocking_pairs': 2.0}




In [ ]:
task3_reasoning_detailed, task3_reasoning_summary = task3_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 3 reasoning summary - 10 instances")
print(task3_reasoning_summary)




task3_reasoning_detailed_20, task3_reasoning_summary_20 = task3_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=20,
    num_examples_to_show=2
)

print()
print("task 3 reasoning summary - 20 instances")
print(task3_reasoning_summary_20)




save_summary_json(task3_reasoning_summary, SUMMARIES_DIR / "task3_reasoning_10.json")
save_summary_json(task3_reasoning_summary_20, SUMMARIES_DIR / "task3_reasoning_20.json")

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 2
blocking pairs: [('M2', 'W3'), ('M3', 'W3')]



RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kn7yhkggfdkvncbwjbca9kqw` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99917, Requested 1136. Please try again in 15m9.791999999s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# =========================================
# 3.6 task 3 quick comparison
# =========================================

print("task 3 comparison - 10 instance")
print(f"basic stable count: {task3_basic_summary['stable_count']} / {task3_basic_summary['total_instances']}")
print(f"reasoning stable count: {task3_reasoning_summary['stable_count']} / {task3_reasoning_summary['total_instances']}")
print(f"basic exact match count: {task3_basic_summary['exact_match_count']} / {task3_basic_summary['total_instances']}")
print(f"reasoning exact match count: {task3_reasoning_summary['exact_match_count']} / {task3_reasoning_summary['total_instances']}")
print(f"basic avg blocking pairs: {task3_basic_summary['avg_blocking_pairs']}")
print(f"reasoning avg blocking pairs: {task3_reasoning_summary['avg_blocking_pairs']}")


print("task 3 comparison - 20 instance")
print(f"basic stable count: {task3_basic_summary_20['stable_count']} / {task3_basic_summary_20['total_instances']}")
print(f"reasoning stable count: {task3_reasoning_summary_20['stable_count']} / {task3_reasoning_summary_20['total_instances']}")
print(f"basic exact match count: {task3_basic_summary_20['exact_match_count']} / {task3_basic_summary_20['total_instances']}")
print(f"reasoning exact match count: {task3_reasoning_summary_20['exact_match_count']} / {task3_reasoning_summary_20['total_instances']}")
print(f"basic avg blocking pairs: {task3_basic_summary_20['avg_blocking_pairs']}")
print(f"reasoning avg blocking pairs: {task3_reasoning_summary_20['avg_blocking_pairs']}")





task 3 comparison
basic stable count: 1 / 10


NameError: name 'task3_reasoning_summary' is not defined

### Task 3 Insights: Resolving Instability

This task required the models to transform an unstable matching into a stable one.

Both models struggled significantly. The basic model produced only 1 stable matching out of 10 instances, while the reasoning model improved slightly to 3 out of 10.

Although most outputs were valid matchings, they often remained unstable, with an average of approximately 1–2 blocking pairs. This indicates that the models were unable to effectively eliminate all instability, even when explicitly instructed to do so.

Compared to Task 2, where models could sometimes recognize instability, this task highlights that *correcting* instability is substantially more difficult than detecting it.

Overall, resolving instability requires deeper algorithmic reasoning, and both models show clear limitations in this setting.

=========================
TASK 4: PREFERENCE REASONING
=========================
Goal: Answer questions based on preference lists of agents.

Output:

- Agent name (for ranking questions)
- YES or NO (for comparison and decision questions)

In [ ]:
# =========================================
# 4.7 run task 4 experiments
# =========================================

task4_basic_detailed, task4_basic_summary = task4_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 4 basic summary - 10 instance")
print(task4_basic_summary)

print("\n" + "=" * 50 + "\n")


task4_basic_detailed_20, task4_basic_summary_20 = task4_basic_model(
    csv_file="5_ic_processed.csv",
    num_instances=20,
    num_examples_to_show=2
)

print()
print("task 4 basic summary - 20 instance")
print(task4_basic_summary_20)

print("\n" + "=" * 50 + "\n")



save_summary_json(task4_basic_summary, SUMMARIES_DIR / "task4_basic_10.json")
save_summary_json(task4_basic_summary_20, SUMMARIES_DIR / "task4_basic_20.json")


parsed: True
predicted: M1
true answer: M1
correct: True
parsed: True
predicted: YES
true answer: NO
correct: False

task 4 basic summary
{'total_instances': 10, 'parsed_count': 10, 'correct_count': 6, 'accuracy': 0.6}




RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kn7yhkggfdkvncbwjbca9kqw` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99862, Requested 1504. Please try again in 19m40.224s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
task4_reasoning_detailed, task4_reasoning_summary = task4_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=10,
    num_examples_to_show=2
)

print()
print("task 4 reasoning summary - 10 instance")
print(task4_reasoning_summary)



task4_reasoning_detailed_20, task4_reasoning_summary_20 = task4_reasoning_model(
    csv_file="5_ic_processed.csv",
    num_instances=20,
    num_examples_to_show=2
)

print()
print("task 4 reasoning summary - 20 instance")
print(task4_reasoning_summary_20)




save_summary_json(task4_reasoning_summary, SUMMARIES_DIR / "task4_reasoning_10.json")
save_summary_json(task4_reasoning_summary_20, SUMMARIES_DIR / "task4_reasoning_20.json")


In [ ]:
# =========================================
# 4.8 task 4 quick comparison
# =========================================

print("task 4 comparison - 10 instances")
print(f"basic correct count: {task4_basic_summary['correct_count']} / {task4_basic_summary['total_instances']}")
print(f"reasoning correct count: {task4_reasoning_summary['correct_count']} / {task4_reasoning_summary['total_instances']}")
print(f"basic accuracy: {task4_basic_summary['accuracy']}")
print(f"reasoning accuracy: {task4_reasoning_summary['accuracy']}")





print("task 4 comparison - 20 instances")
print(f"basic correct count: {task4_basic_summary_20['correct_count']} / {task4_basic_summary_20['total_instances']}")
print(f"reasoning correct count: {task4_reasoning_summary_20['correct_count']} / {task4_reasoning_summary_20['total_instances']}")
print(f"basic accuracy: {task4_basic_summary_20['accuracy']}")
print(f"reasoning accuracy: {task4_reasoning_summary_20['accuracy']}")

### Task 4 Insights: Preference Reasoning

Both models performed better on preference reasoning compared to other tasks. The basic model achieved 60% accuracy, while the reasoning model achieved 70%.

Unlike Tasks 1 and 3, which require enforcing global stability constraints, this task focuses on local reasoning over preference lists. The higher accuracy indicates that LLMs are more effective at answering direct queries about rankings and comparisons.

However, errors still occur, particularly in yes/no comparison questions, suggesting that even local reasoning is not fully reliable.

Overall, this task highlights that LLMs are stronger at interpreting preferences than constructing or correcting stable matchings.

In [ ]:
# load the results from the files created earlier 
summary_map = {
    ("Task 1", "Basic", 10): load_summary_json(SUMMARIES_DIR / "task1_basic_10.json"),
    ("Task 1", "Reasoning", 10): load_summary_json(SUMMARIES_DIR / "task1_reasoning_10.json"),
    ("Task 1", "Basic", 20): load_summary_json(SUMMARIES_DIR / "task1_basic_20.json"),
    ("Task 1", "Reasoning", 20): load_summary_json(SUMMARIES_DIR / "task1_reasoning_20.json"),

    ("Task 2", "Basic", 10): load_summary_json(SUMMARIES_DIR / "task2_basic_10.json"),
    ("Task 2", "Reasoning", 10): load_summary_json(SUMMARIES_DIR / "task2_reasoning_10.json"),
    ("Task 2", "Basic", 20): load_summary_json(SUMMARIES_DIR / "task2_basic_20.json"),
    ("Task 2", "Reasoning", 20): load_summary_json(SUMMARIES_DIR / "task2_reasoning_20.json"),

    ("Task 3", "Basic", 10): load_summary_json(SUMMARIES_DIR / "task3_basic_10.json"),
    ("Task 3", "Reasoning", 10): load_summary_json(SUMMARIES_DIR / "task3_reasoning_10.json"),
    ("Task 3", "Basic", 20): load_summary_json(SUMMARIES_DIR / "task3_basic_20.json"),
    ("Task 3", "Reasoning", 20): load_summary_json(SUMMARIES_DIR / "task3_reasoning_20.json"),

    ("Task 4", "Basic", 10): load_summary_json(SUMMARIES_DIR / "task4_basic_10.json"),
    ("Task 4", "Reasoning", 10): load_summary_json(SUMMARIES_DIR / "task4_reasoning_10.json"),
    ("Task 4", "Basic", 20): load_summary_json(SUMMARIES_DIR / "task4_basic_20.json"),
    ("Task 4", "Reasoning", 20): load_summary_json(SUMMARIES_DIR / "task4_reasoning_20.json"),
}

In [ ]:
final_summary_df = build_final_summary_table(summary_map)
display(final_summary_df)

save_dataframe_csv(final_summary_df, TABLES_DIR / "final_summary_table.csv")

In [ ]:
chart_df = build_chart_dataframe(summary_map)
fig = plot_grouped_bar_chart(chart_df, FIGURES_DIR / "grouped_bar_chart.png")